In [1]:
import torch
import math
from collections import defaultdict
from tokenizers import Tokenizer

In [2]:
best_rl_thinking = '/kaggle/input/datasets/absolute1000/model-after-rl/RL_BEST.pt'
best_rl_classic = '/kaggle/input/datasets/absolute1000/classic-model-after-rl/RL_BEST.pt'
best_sft_thinking = '/kaggle/input/datasets/absolute1000/model-after-sft/SFT_BEST.pt'
best_sft_classic = '/kaggle/input/datasets/absolute1000/classic-model-after-sft/SFT_BEST.pt'

In [4]:
def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN_Classic(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 thinking_internal_state_size, 
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

In [5]:
def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class Thinking(torch.nn.Module):
    def __init__(self, emb_size, internal_state_size=128, vocab_size=12288, momentum=0.1):
        super().__init__()
        self.momentum = momentum
        self.to_vocab = torch.nn.Linear(emb_size, vocab_size, bias=False)
        self.alpha = torch.nn.Parameter(torch.ones(vocab_size))
        self.SiLU = torch.nn.SiLU()

        self.to_internal = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.ln = torch.nn.LayerNorm(internal_state_size)
        self.to_emb = torch.nn.Linear(internal_state_size, emb_size, bias=False)

        self.register_buffer("e_mean", torch.zeros(vocab_size))
        self.t_in = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.t_ln = torch.nn.LayerNorm(internal_state_size)
        self.t_out = torch.nn.Linear(internal_state_size, emb_size, bias=False)

    def forward(self, x, mask_for_thinking, context_ids):
        # x [B, Cs, Emb], mask_for_thinking [B, Cs, 1], context_ids [B, Cs]
        x_unit = x / x.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        vocab_unit = self.to_vocab.weight / self.to_vocab.weight.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        energy = x_unit @ vocab_unit.T # [B, Cs, V], from -1 to 1

        own = torch.zeros_like(energy, dtype=torch.bool).scatter_(2, context_ids[..., None], True)
        drop = (~mask_for_thinking) | own # padding + the position's own token
        kept = energy.masked_fill(drop, 0.0) # [B, Cs, V]

        # words channel
        internal_state = kept * self.alpha
        internal_state = self.to_internal(internal_state)
        internal_state = self.ln(internal_state)
        internal_state = self.SiLU(internal_state)
        words = self.to_emb(internal_state) # [B, Cs, Emb]

        # thought channel
        e = kept.sum(dim=1).float() # [B, V]: sum of energies over positions
        if self.training:
            mu = e.mean(dim=0) # mean over texts in the batch
            with torch.no_grad():
                self.e_mean.mul_(1 - self.momentum).add_(self.momentum * mu)
        else:
            mu = self.e_mean # inference: running mean
        e = e - mu # how this text differs from the others
        e = e / e.pow(2).mean(dim=-1, keepdim=True).sqrt().clamp(min=1e-6) # unit scale

        thought = self.t_in(e)
        thought = self.t_ln(thought)
        thought = self.SiLU(thought)
        thought = self.t_out(thought) # [B, Emb]

        thinking_out = words + thought.unsqueeze(1) # thought is added to every position
        return thinking_out

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN_Thinking(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 thinking_internal_state_size, 
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Thinking = Thinking(embedding_size, thinking_internal_state_size)
        self.Thinking.to_vocab.weight = self.Embedding.weight
        self.out_norm = torch.nn.RMSNorm(embedding_size)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)
        torch.nn.init.zeros_(self.Thinking.t_out.weight)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        memory = self.Thinking(memory, mask.transpose(1, 2), context_ids)
        memory = self.out_norm(memory)
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

In [6]:
TOKENIZER = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/tokenizer.json"
tok = Tokenizer.from_file(TOKENIZER)
USER_ID, BOT_ID = tok.token_to_id("<user>"), tok.token_to_id("<bot>")
PAD_ID, MASK_ID, EOS_ID = tok.token_to_id("<pad>"), tok.token_to_id("<mask>"), tok.token_to_id("<eos>")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_model(path, cls):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    m = cls(**ck["MODEL_SETTINGS_CONFIG"]).to(DEVICE)
    m.load_state_dict(ck["model"])
    return m.eval()


MODELS = {
    "SFT thinking": load_model(best_sft_thinking, NN_Thinking),
    "RL thinking":  load_model(best_rl_thinking, NN_Thinking),
    "SFT classic":  load_model(best_sft_classic, NN_Classic),
    "RL classic":   load_model(best_rl_classic, NN_Classic),
}
print("загружено:", ", ".join(MODELS))

загружено: SFT thinking, RL thinking, SFT classic, RL classic


In [9]:
def trim_history(history, context_len=256):
    if len(history) <= context_len:
        return history
    for i, t in enumerate(history):
        if t == USER_ID and len(history) - i <= context_len:
            return history[i:]
    return history[-context_len:]


def ngram_tables(seq, n):
    nxt, prv = defaultdict(set), defaultdict(set)
    for j in range(len(seq) - n + 1):
        g = seq[j:j + n]
        if MASK_ID in g:
            continue
        nxt[tuple(g[:-1])].add(g[-1])
        prv[tuple(g[1:])].add(g[0])
    return nxt, prv


def ngram_banned(a, i, L, nxt, prv, n):
    m, banned = n - 1, set()
    if i >= m and MASK_ID not in a[i - m:i]:
        banned |= nxt.get(tuple(a[i - m:i]), set())
    if i + m < L and MASK_ID not in a[i + 1:i + 1 + m]:
        banned |= prv.get(tuple(a[i + 1:i + 1 + m]), set())
    return banned


@torch.no_grad()
def fill(model, ans, memory, mask, region, ctx_ids, L, temperature, top_k, block_size, rep_penalty,
         ngram_ans, ngram_ctx, ngram_penalty):
    pos = torch.arange(ans.shape[1], device=ans.device)
    ctx_tables = ngram_tables(ctx_ids, ngram_ctx) if ngram_ctx > 1 else None
    while True:
        is_mask = ans[0] == MASK_ID
        todo = is_mask & region
        if not todo.any():
            break
        logits = model.decode(ans, memory, mask)[0].float()
        logits[:, [MASK_ID, PAD_ID, EOS_ID]] = -float("inf")
        used = ans[0][~is_mask].unique()
        used = used[(used != EOS_ID) & (used != PAD_ID)]
        if rep_penalty > 1.0 and len(used):
            lu = logits[:, used]
            logits[:, used] = torch.where(lu > 0, lu / rep_penalty, lu * rep_penalty)
        first = int(todo.nonzero()[0])
        eligible = todo & (pos < (first // block_size + 1) * block_size)
        if ngram_penalty > 0:
            a = ans[0].tolist()
            ans_tables = ngram_tables(a[:L], ngram_ans) if ngram_ans > 1 else None
            for i in eligible.nonzero().squeeze(-1).tolist():
                banned = set()
                if ans_tables:
                    banned |= ngram_banned(a, i, L, *ans_tables, ngram_ans)
                if ctx_tables:
                    banned |= ngram_banned(a, i, L, *ctx_tables, ngram_ctx)
                if banned:
                    logits[i, list(banned)] -= ngram_penalty
        logits = logits / temperature
        kth = logits.topk(top_k, dim=-1).values[:, -1:]
        probs = torch.softmax(logits.masked_fill(logits < kth, -float("inf")), -1)
        pred = torch.multinomial(probs, 1).squeeze(-1)
        conf = probs.gather(-1, pred[:, None]).squeeze(-1).masked_fill(~eligible, -1.0)
        i = int(conf.argmax())
        ans[0, i] = pred[i]


@torch.no_grad()
def loo_probs(model, ans, memory, mask, end):
    batch = ans.repeat(end, 1)
    idx = torch.arange(end, device=ans.device)
    batch[idx, idx] = MASK_ID
    logits = model.decode(batch, memory.expand(end, -1, -1), mask.expand(end, -1, -1)).float()[idx, idx]
    logits[:, [MASK_ID, PAD_ID, EOS_ID]] = -float("inf")
    return torch.softmax(logits, -1)


@torch.no_grad()
def gstar(model, ans, memory, mask, start, end, rounds, margin, protected):
    for _ in range(rounds):
        probs = loo_probs(model, ans, memory, mask, end)
        cur = ans[0, :end]
        p_cur = probs.gather(-1, cur[:, None]).squeeze(-1)
        p_best, best_tok = probs.max(-1)
        bad = (p_cur < margin * p_best) & (best_tok != cur)
        bad[:start] = False
        bad &= ~torch.isin(cur, protected)
        if not bad.any():
            break
        j = int(torch.where(bad, p_best - p_cur, torch.full_like(p_cur, -1.0)).argmax())
        ans[0, j] = best_tok[j]


@torch.no_grad()
def elite_generate(model, ids, answer_len=64, temperature=0.6, top_k=50, block_size=8, rep_penalty=1.05,
                   min_len=3, length_scale=1.0, n_candidates=4, block_rounds=2, global_rounds=2,
                   margin=0.5, ngram_ans=3, ngram_ctx=4, ngram_penalty=5.0):
    ctx = torch.tensor([ids], device=DEVICE)
    memory, mask = model.encode(ctx)
    base = torch.full((1, answer_len), MASK_ID, dtype=torch.long, device=DEVICE)
    probs0 = torch.softmax(model.decode(base, memory, mask)[0].float(), -1)
    L_mean = (1 - probs0[:, EOS_ID] - probs0[:, PAD_ID]).sum().item() * length_scale
    pos = torch.arange(answer_len, device=DEVICE)

    u = len(ids) - 1 - ids[::-1].index(USER_ID) if USER_ID in ids else -1
    protected = torch.tensor(ids[u + 1:-1] if u >= 0 else [], dtype=torch.long, device=DEVICE)

    scales = torch.linspace(0.8, 1.2, n_candidates).tolist() if n_candidates > 1 else [1.0]
    best_ids, best_score = None, -float("inf")
    for s in scales:
        L = max(min_len, min(int(round(L_mean * s)), answer_len - 1))
        ans = base.clone()
        ans[0, L] = EOS_ID
        ans[0, L + 1:] = PAD_ID
        fill_args = dict(ctx_ids=ids, L=L, temperature=temperature, top_k=top_k, block_size=block_size,
                         rep_penalty=rep_penalty, ngram_ans=ngram_ans, ngram_ctx=ngram_ctx,
                         ngram_penalty=ngram_penalty)
        for b_start in range(0, L, block_size):
            b_end = min(b_start + block_size, L)
            fill(model, ans, memory, mask, region=pos < b_end, **fill_args)
            gstar(model, ans, memory, mask, b_start, b_end, block_rounds, margin, protected)
        gstar(model, ans, memory, mask, 0, L, global_rounds, margin, protected)

        p = loo_probs(model, ans, memory, mask, L).gather(-1, ans[0, :L, None]).squeeze(-1)
        score = p.clamp_min(1e-9).log().mean().item()
        if score > best_score:
            best_score, best_ids = score, ans[0, :L].tolist()

    return tok.decode(best_ids), best_ids, best_score

In [8]:
def chat(models=MODELS, mode="dialog", context_len=256, seed=0, **gen_kwargs):
    hist = {name: [] for name in models}
    print(f"режим: {mode}, окно {context_len}. Выход — Enough")
    while (a := input()) != "Enough":
        print(f"\nUSER: {a}")
        for name, m in models.items():
            if mode == "single":
                hist[name] = []
            hist[name] += [USER_ID] + tok.encode(a).ids + [BOT_ID]
            window = trim_history(hist[name], context_len)
            torch.manual_seed(seed)
            reply, reply_ids, score = elite_generate(m, window, **gen_kwargs)
            hist[name] += reply_ids + [EOS_ID]
            print(f"  {name:>13} ({score:+.2f}): {reply}")

In [12]:
chat(mode="single", n_candidates=8, margin=0.3, temperature=0.5)

режим: single, окно 256. Выход — Enough


 Capital of France?



USER: Capital of France?
   SFT thinking (-0.28): Yes, there is a trade trade.
    RL thinking (-0.48): No,No,No,No,No,No,No.
    SFT classic (-0.07): Sure.
     RL classic (-0.14): Sure.


 What is capital of France?



USER: What is capital of France?
   SFT thinking (-0.06): Indeed.
    RL thinking (-0.36): I agree.
    SFT classic (-0.53): It's leading capital of France. It takes a lot of money.
     RL classic (-0.64): Good choice.


...

 You prefer football or basketball?



USER: You prefer football or basketball?
   SFT thinking (-0.14): Yes, I do.
    RL thinking (-0.45): Yes, I do not have a team or team or team.
    SFT classic (-0.03): Of course.
     RL classic (-0.07): Of course.


 Do you like computer games?



USER: Do you like computer games?
   SFT thinking (-0.06): Sure.
    RL thinking (-0.22): Oh, yes, I do.
    SFT classic (-0.03): Of course.
     RL classic (-0.04): Of course.


KeyboardInterrupt: Interrupted by user

In [13]:
chat(mode="dialog", n_candidates=8, margin=0.3, temperature=0.5)

режим: dialog, окно 256. Выход — Enough


 Hello.



USER: Hello.
   SFT thinking (-0.21): Hi. Hello, how are you today?
    RL thinking (-0.28): HelloHHello, how are you doing?
    SFT classic (-0.28): Hi, I'm sorry.
     RL classic (-0.07): Hello.


 I am going to play football after breakfast.



USER: I am going to play football after breakfast.
   SFT thinking (-0.18): Do you have any hobbies? I am a big fan of baseball.
    RL thinking (-0.42): I am a home nurse. I am a nurse. How are you?
    SFT classic (-0.16): I'm going to play football.
     RL classic (-0.02): Hello.


 Do you want to go to play football with me?



USER: Do you want to go to play football with me?
   SFT thinking (-0.07): I do not have any hobbies. Do you?
    RL thinking (-0.30): I am a nurse. I am a vegetarian nurse, thank you.
    SFT classic (-0.13): I'm not sure.
     RL classic (-0.01): Hello.


 Ok. See you on the field near the shop. Goodbye.



USER: Ok. See you on the field near the shop. Goodbye.
   SFT thinking (-0.08): Do you have any hobbies? How about you?
    RL thinking (-0.18): Cool. I am a librarian. I do not want to be a librarian.
    SFT classic (-0.03): Sure. Thank you.
     RL classic (-0.01): Hello.


KeyboardInterrupt: Interrupted by user

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM

PYTHIA_NAME = "EleutherAI/pythia-14m"
py_tok = AutoTokenizer.from_pretrained(PYTHIA_NAME)
py_model = AutoModelForCausalLM.from_pretrained(PYTHIA_NAME).to(DEVICE).eval()
print(f"Pythia: {sum(p.numel() for p in py_model.parameters()) / 1e6:.1f}M параметров")

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.83k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 28.1MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Pythia: 14.1M параметров


In [15]:
@torch.no_grad()
def pythia_reply(prompt, max_new_tokens=64):
    inp = py_tok(prompt, return_tensors="pt").to(DEVICE)
    out = py_model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                            temperature=0.7, top_k=50, repetition_penalty=1.3,
                            pad_token_id=py_tok.eos_token_id)
    text = py_tok.decode(out[0, inp.input_ids.shape[1]:], skip_special_tokens=True)
    return text.split("\n")[0].split("User:")[0].strip()


def chat_pythia(mode="dialog", context_len=256, seed=0):
    history = ""
    print(f"Pythia-14M, режим: {mode}. Выход — Enough")
    while (a := input()) != "Enough":
        if mode == "single":
            history = ""
        history += f"User: {a}\n"
        ids = py_tok(history).input_ids[-context_len:]
        prompt = py_tok.decode(ids) + "Bot:"
        torch.manual_seed(seed)
        reply = pythia_reply(prompt)
        history += f"Bot: {reply}\n"
        print(f"\nUSER: {a}\n  Pythia-14M: {reply}")

In [16]:
chat_pythia(mode="dialog")

Pythia-14M, режим: dialog. Выход — Enough


 Hello.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPTNeoXTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



USER: Hello.
  Pythia-14M: google/Google Apps/Mac, Google AppDomain


 I am going to play football after breakfast.


...

 Do you want to go to play football with me?



USER: Do you want to go to play football with me?
  Pythia-14M: google/google+apolqiwkijhkluhsaketkuhoanokuyaaamahajukkohujihazama


 Ok. See you on the field near the shop. Goodbye.



USER: Ok. See you on the field near the shop. Goodbye.
  Pythia-14M: google/google+apolqwdwkiuyaojiakkibioohahtuahi jaaatjujakainhujimazotuihesee kamiestenio takoštiimusiejo zljeđeni jeli ukožjemp


KeyboardInterrupt: Interrupted by user

In [17]:
GPT2_NAME = "openai-community/gpt2"
g2_tok = AutoTokenizer.from_pretrained(GPT2_NAME)
g2_model = AutoModelForCausalLM.from_pretrained(GPT2_NAME).to(DEVICE).eval()
print(f"GPT-2: {sum(p.numel() for p in g2_model.parameters()) / 1e6:.1f}M параметров")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2: 124.4M параметров


In [18]:
@torch.no_grad()
def gpt2_reply(prompt, max_new_tokens=64):
    inp = g2_tok(prompt, return_tensors="pt").to(DEVICE)
    out = g2_model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                            temperature=0.7, top_k=50, repetition_penalty=1.3,
                            pad_token_id=g2_tok.eos_token_id)
    text = g2_tok.decode(out[0, inp.input_ids.shape[1]:], skip_special_tokens=True)
    return text.split("\n")[0].split("User:")[0].strip()


def chat_gpt2(mode="dialog", context_len=256, seed=0):
    history = ""
    print(f"GPT-2, режим: {mode}. Выход — Enough")
    while (a := input()) != "Enough":
        if mode == "single":
            history = ""
        history += f"User: {a}\n"
        ids = g2_tok(history).input_ids[-context_len:]
        prompt = g2_tok.decode(ids) + "Bot:"
        torch.manual_seed(seed)
        reply = gpt2_reply(prompt)
        history += f"Bot: {reply}\n"
        print(f"\nUSER: {a}\n  GPT-2: {reply}")

In [19]:
chat_gpt2(mode="dialog")

GPT-2, режим: dialog. Выход — Enough


 Hello.



USER: Hello.
  GPT-2: How are you? If I can help, please send me a message so i know where to find your address and how much is there for my account! [02/01 14] PM #16 rick_jones Bot: Okay guys.. do we have any more questions or needs at this point!? Thank you


 I am going to play football after breakfast.



USER: I am going to play football after breakfast.
  GPT-2: So what's up with that red shirt color???? ? Hey whats the difference between blue t-shirt & navy purple jersey right now??? It was always green shirts....I think it looked better on them but they need some work...what happened here??


 Do you want to go to play football with me?



USER: Do you want to go to play football with me?
  GPT-2: No :(


 Ok. See you on the field near the shop. Goodbye.



USER: Ok. See you on the field near the shop. Goodbye.
  GPT-2: Bye :)


KeyboardInterrupt: Interrupted by user